In [0]:
# Fetch the key at runtime — this value never appears in the notebook source
api_key = dbutils.secrets.get(scope="census_api", key="acs_key")

# Use it normally in your Census API call
import requests
import pandas as pd

# details of subject tables availalbe - https://www.census.gov/acs/www/data/data-tables-and-tools/subject-tables/
subject_groups = ["S2701",              # Health insurance coverage status
                  "S2704",              # Public Health insurance coverage type
                  "S2703",              # Private Health Insurance coverage type 
                  "S1101",              # Households and Families
                  "S1201",              # Marital Status
                  "S1002",              # Grandparents
                  "S1701",              # Poverty Status in the Past 12 Months (individual)
                  "S1702",              # Poverty Status in the Past 12 Months of Families
                  "S2303",               # Employment Characteristics of Families
                  "S2501",              # Occupancy Characteristics
                  "S0801",              # Commuting Characteristics (includes travel to work)
                  "S2503",              # Financial characteristics (houshold income + housing cost)
                  "S9801",              # Quality Standard Metrics (Marital status, )
                  ]

dfs = {}

for group in subject_groups:
    url = "https://api.census.gov/data/2024/acs/acs5/subject"
    

    params = {
        "get": f"NAME.group({group})",                     # Health insurance coverage status
        "for": "tract:*",
        "in": "state:06 county:*",
        "key": api_key          # injected at runtime, not hardcoded
    }

    response = requests.get(url, params=params)

    print(f"processed {group} with Status code: {response.status_code}")

    group_data = response.json()
    pd_df = pd.DataFrame(group_data)
    
    dfs[group] = pd_df

    print(f"Fetched {group}: {pd_df.shape}")
#})
# DBTITLE 1,Save the DataFrames t



Drop the Margin of error and annotation columns and keep only the estimate columns with suffix "E"

In [0]:
geo_cols = ['state', 'county', 'tract', 'GEO_ID', 'NAME']

def filter_estimate_cols(df, geo_cols):
    # Set the first row column label
    df = df.rename(columns=df.iloc[0]).drop(df.index[0]).reset_index(drop=True)
    cols = df.columns
    cols_to_keep = [col for col in cols if col.endswith("E") or col in geo_cols]
    return df[cols_to_keep]


In [0]:
df_clean = {}

for group, df in dfs.items():
    df_clean[group] = filter_estimate_cols(df, geo_cols)
    print(f"Cleaned {group}: shapes - {df_clean[group].shape} columns - {df_clean[group].columns}")



Convert pandas dataframes to spark df and store in schema

In [0]:
for key, value in df_clean.items():
    spark_df = spark.createDataFrame(value)
    table_name = f"ca_healthcare_fac_bronze.census_data_ingest_bronze.{key}"
    spark_df.write.mode("overwrite").saveAsTable(table_name)
    print(f"Wrote {key} to {table_name}")
 